In [5]:
#pip install wikipedia
#%pip install sentence-transformers
#%pip install faiss-cpu

In [ ]:
from langchain_community.document_loaders import WikipediaLoader #o fetch the article
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS #embeddings for FAISS.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough #to forward the question into the chain
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceBgeEmbeddings


#### INDEXING ####
# Load

loader = WikipediaLoader(
    query="Retrieval-augmented generation",
    lang="en",
    load_max_docs=1,
    doc_content_chars_max=200000,
)
docs = loader.load()

print(len(docs), docs[0].metadata)
print(docs[0].page_content[:600])  # clean article text

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = splitter.split_documents(docs)
print("Document splits:", len(splits))


#Embed
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hugging_embeddings = HuggingFaceBgeEmbeddings(
     model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
 )
vectorstore = FAISS.from_documents(documents=splits, 
                                    embedding=hugging_embeddings)

retriever = vectorstore.as_retriever() # Dense Retrieval - Embeddings/Context based

# #### RETRIEVAL and GENERATION ####

# # Prompt
prompt = ChatPromptTemplate.from_template(
    """You are a helpful AI assistant. 
Explain the answer as if you are teaching a young student, 
using very simple words and short sentences. 

Only use the following retrieved context to answer the question. 
If the context does not contain the answer, say "I don’t know."

Context:
{context}

Question:
{question}

Answer:
"""
)

# # LLM

llm = Ollama(model="llama3.2", temperature=0)
# # Post-processing
def format_wiki(docs):
     return "\n\n".join(doc.page_content for doc in docs)

# # Chain
rag_chain = (
    {"context": retriever | format_wiki, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# # Question
print(rag_chain.invoke("What is RAG"))

1 {'title': 'Retrieval-augmented generation', 'summary': 'Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information. With RAG, LLMs do not respond to user queries until they refer to a specified set of documents. These documents supplement information from the LLM\'s pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this helps LLM-based chatbots access internal company data or generate responses based on authoritative sources.\nRAG improves large language models (LLMs) by incorporating information retrieval before generating responses. Unlike traditional LLMs that rely on static training data, RAG pulls relevant text from databases, uploaded documents, or web sources. According to Ars Technica, "RAG is a way of improving LLM performance, in essence by blending the LLM process with a web search or other docu